# Step 1: 데이터 준비

## 학습 목표
이 노트북을 완료하면 다음을 이해할 수 있습니다:
- **딥페이크 탐지**를 위한 데이터셋 구조
- **Train/Validation/Test** 분할의 목적과 중요성
- SageMaker에서 S3를 활용한 데이터 관리 방법

## 배경 지식

### 딥페이크(Deepfake)란?
AI를 사용하여 사람의 얼굴이나 음성을 합성하는 기술입니다. 
- **Real**: 실제 촬영된 얼굴 이미지
- **Fake**: AI(StyleGAN 등)로 생성된 가짜 얼굴 이미지

### 왜 Fine-tuning이 필요한가?
기존 딥페이크 탐지 모델(FaceForensics++)은 **서양인 얼굴** 위주로 학습되어 있어,
**한국인 얼굴**에서는 탐지 성능이 떨어집니다. 이를 개선하기 위해 Fine-tuning을 수행합니다.

### 데이터 분할 전략
| 구분 | 비율 | 용도 |
|------|------|------|
| **Train** | 70% | 모델 학습에 사용 |
| **Validation** | 15% | 학습 중 과적합 모니터링 |
| **Test** | 15% | 최종 성능 평가 (학습에 절대 사용 안 함) |

> ⚠️ **중요**: Test 데이터는 Before/After 비교를 위해 **동일하게** 유지해야 합니다.

## 1.1 환경 설정

SageMaker 환경을 초기화하고 필요한 변수들을 설정합니다.

**확인할 것들:**
- `Role`: SageMaker가 AWS 리소스에 접근할 수 있는 IAM 역할
- `Bucket`: 데이터와 모델이 저장될 S3 버킷 (자동 생성됨)
- `Region`: 현재 작업 중인 AWS 리전

In [ ]:
import os
import sys
import boto3
import sagemaker
from pathlib import Path

# ============================================
# 프로젝트 경로 자동 설정
# ============================================
# 홈 디렉토리에서 프로젝트 폴더 찾기
home_dir = Path.home()
PROJECT_ROOT = home_dir / 'deepfake-detection-sagemaker'

# 현재 노트북 디렉토리로 이동
notebook_dir = PROJECT_ROOT / '1_data_preparation'
os.chdir(notebook_dir)

print(f"Home: {home_dir}")
print(f"Project Root: {PROJECT_ROOT}")
print(f"Current Dir: {os.getcwd()}")

# SageMaker 세션 설정
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session.boto_region_name

# S3 버킷 설정
bucket = sagemaker_session.default_bucket()
prefix = 'deepfake-detection'

print(f"\nRegion: {region}")
print(f"Role: {role[:50]}...")
print(f"Bucket: {bucket}")

## 1.2 Workshop 데이터 다운로드

미리 준비된 딥페이크 샘플 데이터를 S3에서 다운로드합니다.

### 데이터셋 정보
- **출처**: 140k Real and Fake Faces (Kaggle)
- **Real 이미지**: 실제 얼굴 사진
- **Fake 이미지**: StyleGAN으로 생성된 가짜 얼굴

### 데이터 구성
| 구분 | Real | Fake | 합계 | 용도 |
|------|------|------|------|------|
| Train | 1,000장 | 1,000장 | 2,000장 | 모델 학습 |
| Validation | 200장 | 200장 | 400장 | 과적합 모니터링 |
| Test | 200장 | 200장 | 400장 | **Before/After 비교** |

> 💡 **포인트**: Test 데이터는 Fine-tuning 전후 성능을 **공정하게 비교**하기 위해 동일하게 유지합니다.

In [ ]:
from pathlib import Path
import subprocess
import os

# 데이터 디렉토리 설정
data_dir = Path('./data')
data_dir.mkdir(exist_ok=True)

# Workshop 데이터 소스 (CloudFront)
CLOUDFRONT_URL = "https://d291vm7e8ubihi.cloudfront.net"
DATA_SOURCE = f"{CLOUDFRONT_URL}/sample-data"

print(f"데이터 소스: {DATA_SOURCE}")
print("데이터 다운로드 중... (약 1-2분 소요)\n")

# 디렉토리 구조 생성
for split in ['train', 'val', 'test']:
    for label in ['real', 'fake']:
        (data_dir / split / label).mkdir(parents=True, exist_ok=True)

# 데이터 다운로드 함수
def download_images(split, label, count):
    """CloudFront에서 이미지 다운로드"""
    target_dir = data_dir / split / label
    downloaded = 0
    
    for i in range(1, count + 1):
        filename = f"{i:04d}.jpg"
        url = f"{DATA_SOURCE}/{split}/{label}/{filename}"
        target_path = target_dir / filename
        
        result = subprocess.run(
            ['curl', '-sf', '-o', str(target_path), url],
            capture_output=True
        )
        
        if result.returncode == 0:
            downloaded += 1
        
        # 진행률 표시 (100개마다)
        if i % 100 == 0:
            print(f"  {split}/{label}: {i}/{count} 완료")
    
    return downloaded

# 데이터 구성: {split: {label: count}}
data_config = {
    'train': {'real': 1000, 'fake': 1000},
    'val': {'real': 200, 'fake': 200},
    'test': {'real': 200, 'fake': 200}
}

# 다운로드 실행
total_downloaded = 0
for split, labels in data_config.items():
    print(f"\n📥 {split} 데이터 다운로드 중...")
    for label, count in labels.items():
        downloaded = download_images(split, label, count)
        total_downloaded += downloaded
        print(f"  ✅ {split}/{label}: {downloaded}장")

print(f"\n✅ 다운로드 완료! (총 {total_downloaded}장)")

In [ ]:
# 데이터 구조 확인
print("데이터 디렉토리 구조:")
!find ./data -type d

print("\n데이터 개수 확인:")
total = 0
for split in ['train', 'val', 'test']:
    split_dir = data_dir / split
    if split_dir.exists():
        real_count = len(list((split_dir / 'real').glob('*')))
        fake_count = len(list((split_dir / 'fake').glob('*')))
        print(f"{split}: Real={real_count}, Fake={fake_count}, Total={real_count+fake_count}")
        total += real_count + fake_count

print(f"\n총 이미지 수: {total}장")

## 1.3 샘플 이미지 확인

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

# 샘플 이미지 시각화
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i, label in enumerate(['real', 'fake']):
    label_dir = data_dir / 'test' / label
    if label_dir.exists():
        images = list(label_dir.glob('*.jpg')) + list(label_dir.glob('*.png'))
        samples = random.sample(images, min(4, len(images)))
        
        for j, img_path in enumerate(samples):
            img = Image.open(img_path)
            axes[i, j].imshow(img)
            axes[i, j].axis('off')
            axes[i, j].set_title(f"{label.upper()}")

plt.suptitle("Sample Images (Real vs Fake)", fontsize=14)
plt.tight_layout()
plt.show()

## 1.4 내 S3 버킷에 업로드

다운로드한 데이터를 본인의 S3 버킷에 업로드합니다.

In [ ]:
# S3에 데이터 업로드
print(f"내 S3 버킷에 업로드 중: s3://{bucket}/{prefix}/data/")

s3_data_path = sagemaker_session.upload_data(
    path='./data',
    bucket=bucket,
    key_prefix=f'{prefix}/data'
)

print(f"\n✅ 업로드 완료: {s3_data_path}")

## 1.5 설정 저장

다음 노트북에서 사용할 설정을 저장합니다.

In [ ]:
import json

config = {
    'bucket': bucket,
    'prefix': prefix,
    'project_root': str(PROJECT_ROOT),
    's3_data_path': s3_data_path,
    's3_train_path': f's3://{bucket}/{prefix}/data/train',
    's3_val_path': f's3://{bucket}/{prefix}/data/val',
    's3_test_path': f's3://{bucket}/{prefix}/data/test',
    'local_test_path': str(data_dir.absolute() / 'test'),
    'role': role,
    'region': region
}

# 프로젝트 루트에 config.json 저장
config_path = PROJECT_ROOT / 'config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"✅ 설정 저장 완료: {config_path}")
print("\n저장된 설정:")
print(json.dumps(config, indent=2))

## ✅ 완료!

데이터 준비가 완료되었습니다.

**확인 사항:**
- [x] 딥페이크 데이터 다운로드 완료
- [x] Train/Val/Test 데이터 확인
- [x] 내 S3 버킷에 업로드 완료
- [x] config.json 저장 완료

---

**➡️ 다음 단계: `2_before_evaluation/evaluate_before.ipynb`**

Fine-tuning 전 모델의 성능을 평가합니다.